# VunaLink: retrain the tomato model on Kaggle

Trains the 4-class tomato model (**early blight, late blight, healthy, other**), calibrates its confidence for the
"unsure, ask an agent" flow, evaluates it on lab images (PlantVillage) **and** field images (PlantDoc), and exports ONNX for the app.

**Before running:** in the notebook settings turn on **Accelerator → GPU** and **Internet → On**.
Optionally add the PlantVillage and PlantDoc datasets as notebook inputs ("Add Input"); if they're not attached,
the notebook downloads them (PlantVillage through TensorFlow Datasets, PlantDoc from its GitHub repo).
If you attach PlantVillage, attach **one original copy only** (colour images). Augmented copies such as
"New Plant Diseases Dataset (Augmented)" would leak near-duplicate images between train and test.

Run all cells top to bottom. Everything is written to `/kaggle/working`, and the last cell zips what you need for the app.

In [ ]:
REPO = "https://github.com/GabbyIT-pixel/vunalink"
BRANCH = "tomato-4class-retrain"   # change to "main" once this branch is merged
MODEL_VERSION = "0.4.0"            # bump for every model you ship
FIELD_TRAINING = True              # mix PlantDoc field photos into training (v0.3.0+)
QUANTIZE = True                    # also build an INT8 model and report its accuracy

WORK = "/kaggle/working"

In [ ]:
import os, shutil, subprocess
os.chdir(WORK)
shutil.rmtree("vunalink", ignore_errors=True)  # always start from the latest code on GitHub
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "vunalink"], check=True)
os.chdir(f"{WORK}/vunalink")
!git log --oneline -1

Kaggle already has TensorFlow. Don't `pip install -r ml/requirements.txt` here: it can change NumPy under Kaggle's
TensorFlow. Only the export tools are needed.

In [ ]:
!pip install -q tf2onnx onnxruntime
import tensorflow as tf, numpy as np
print("TensorFlow", tf.__version__, "| NumPy", np.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Find or download the data

In [ ]:
import sys
sys.path.insert(0, "ml")
from pathlib import Path
from common import find_class_dirs

def find_input(class_name):
    # Return /kaggle/input if an attached dataset contains this class folder.
    if Path("/kaggle/input").exists() and find_class_dirs("/kaggle/input", [class_name])[class_name]:
        return "/kaggle/input"
    return None

PLANTVILLAGE = find_input("Tomato___Early_blight")
if PLANTVILLAGE is None:
    print("PlantVillage not attached: downloading through TensorFlow Datasets (about 10 minutes)...")
    !python ml/download_plantvillage.py --data-dir {WORK}/tfds --output-dir {WORK}/plant_village
    PLANTVILLAGE = f"{WORK}/plant_village"

PLANTDOC = find_input("Tomato leaf late blight")
if PLANTDOC is None:
    print("PlantDoc not attached: cloning it from GitHub...")
    !git clone --depth 1 -q https://github.com/pratikkayal/PlantDoc-Dataset {WORK}/plantdoc
    PLANTDOC = f"{WORK}/plantdoc"
print("PlantVillage:", PLANTVILLAGE, "| PlantDoc:", PLANTDOC)

## 2. Build the datasets

"Other" means **anything the app doesn't diagnose**: the 7 other tomato diseases, potato, other crops and photos
without leaves. The app sends all of it to an extension agent.

`prepare_field_eval.py` maps PlantDoc's tomato folders, plus its potato, squash powdery mildew and bell pepper folders
(as "other"), onto the 4 classes and splits them 60/15/25 into field train / validation / **test**. The field test
photos are never trained on.

`prepare_dataset.py` builds the 4 classes from PlantVillage ("other" = half other tomato diseases, a quarter potato,
a quarter other crops and photos without leaves), removes duplicates and splits 70/15/15. With `FIELD_TRAINING`, the field train and validation photos are
added too, each field training photo three times so the model can't ignore them. The PlantVillage test split stays
lab-only so the lab number is comparable between versions.

In [ ]:
!python ml/prepare_field_eval.py --source-dir "{PLANTDOC}" --output-dir {WORK}/field
FIELD_ARG = f"--field-dir {WORK}/field" if FIELD_TRAINING else ""
!python ml/prepare_dataset.py --source-dir "{PLANTVILLAGE}" --output-dir {WORK}/split {FIELD_ARG}

## 3. Train

Phase 1 trains the new head, phase 2 fine-tunes the top of MobileNetV2, then the confidence is temperature-calibrated
and the "unsure" threshold is picked on validation data. Settings are in `ml/class_config.json`.
Up to 30 + 30 epochs, with early stopping when validation stops improving. Expect roughly 30–60 minutes on a Kaggle GPU.
If you'll leave the tab, use **Save Version → Save & Run All (Commit)** so it runs in the background.

In [ ]:
!python ml/train.py --data-dir {WORK}/split --output-dir {WORK}/artifacts --version {MODEL_VERSION}

## 4. Evaluate, and compare with the model in the app

The new model on the field test photos, then the model currently in the app (`public/models`, v0.3.0) on exactly
the same lab and field test photos, so the comparison is fair.

In [ ]:
!python ml/evaluate.py --model {WORK}/artifacts/model.keras --data-dir {WORK}/field/test --metadata {WORK}/artifacts/model_metadata.json --dataset-name PlantDoc --output {WORK}/artifacts/evaluation_field.json
!python ml/evaluate.py --model public/models/vunalink-mobilenet-v2.onnx --data-dir {WORK}/split/test --metadata public/models/model_metadata.json --dataset-name "PlantVillage (app model)" --output {WORK}/artifacts/baseline_app_lab.json
!python ml/evaluate.py --model public/models/vunalink-mobilenet-v2.onnx --data-dir {WORK}/field/test --metadata public/models/model_metadata.json --dataset-name "PlantDoc (app model)" --output {WORK}/artifacts/baseline_app_field.json

In [ ]:
import json, pandas as pd
lab = json.load(open(f"{WORK}/artifacts/evaluation.json"))
baseline = json.load(open(f"{WORK}/artifacts/baseline_app_field.json"))
baseline_lab = json.load(open(f"{WORK}/artifacts/baseline_app_lab.json"))
APP_VERSION = json.load(open("public/models/model_metadata.json"))["version"]
field = json.load(open(f"{WORK}/artifacts/evaluation_field.json"))
meta = json.load(open(f"{WORK}/artifacts/model_metadata.json"))
classes = meta["classes"]
print("Temperature:", round(meta["calibration"]["temperature"], 3), "| unsure threshold:", meta["calibration"]["unsure_threshold"])
print("Training:", {k: meta["training"][k] for k in ["head", "fine_tune"]})

summary = pd.DataFrame({
    "PlantVillage test (lab)": {**lab["metrics"], **{k: v for k, v in lab["unsure_routing"].items() if k != "threshold"}, "images": lab["image_count"]},
    "PlantDoc field test (this model)": {**field["metrics"], **{k: v for k, v in field["unsure_routing"].items() if k != "threshold"}, "images": field["image_count"]},
    f"PlantVillage test (app v{APP_VERSION})": {**baseline_lab["metrics"], **{k: v for k, v in baseline_lab["unsure_routing"].items() if k != "threshold"}, "images": baseline_lab["image_count"]},
    f"PlantDoc field test (app v{APP_VERSION})": {**baseline["metrics"], **{k: v for k, v in baseline["unsure_routing"].items() if k != "threshold"}, "images": baseline["image_count"]},
}).round(3)
display(summary)

per_class = pd.DataFrame({
    "lab F1": {c["label"]: c["f1"] for c in lab["per_class"]},
    "field F1": {c["label"]: c["f1"] for c in field["per_class"]},
    f"field F1 app v{APP_VERSION}": {c["label"]: c["f1"] for c in baseline["per_class"]},
    "field support": {c["label"]: c["support"] for c in field["per_class"]},
}).round(3)
display(per_class)
display(pd.DataFrame(field["confusion_matrix"], index=[f"true {c}" for c in classes], columns=[f"pred {c}" for c in classes]))

# The question this version answers: are potato and other non-tomato photos sent to an agent?
routing = pd.DataFrame({
    "lab, this model": {k: v["routed_to_agent"] for k, v in lab["other_routing"]["by_group"].items()},
    f"lab, app v{APP_VERSION}": {k: v["routed_to_agent"] for k, v in baseline_lab["other_routing"]["by_group"].items()},
    "field, this model": {k: v["routed_to_agent"] for k, v in field["other_routing"]["by_group"].items()},
    f"field, app v{APP_VERSION}": {k: v["routed_to_agent"] for k, v in baseline["other_routing"]["by_group"].items()},
    "field images": {k: v["images"] for k, v in field["other_routing"]["by_group"].items()},
}).round(3)
print("Share of 'other' photos sent to an agent (higher is better):")
display(routing)

**Checking the threshold.** With field photos in training, the threshold is chosen on the ~110 field validation photos
(at least 80% of answers right), or on lab photos if that's stricter. Field validation is small, so check the sweep
below on the field **test** photos: if the chosen threshold still gives many wrong answers, raise it. Change
`unsure_threshold` in `model_metadata.json` and `UNSURE_THRESHOLD` in the app's `lib/ml/model.ts` together.

In [ ]:
display(pd.DataFrame(field["threshold_sweep"]).round(3))

## 5. Export to ONNX (and optionally INT8)

In [ ]:
!python ml/export_onnx.py --model {WORK}/artifacts/model.keras --saved-model {WORK}/artifacts/saved_model --output {WORK}/artifacts/vunalink-mobilenet-v2.onnx --check-dir {WORK}/split/test
if QUANTIZE:
    !pip install -q sympy
    !python ml/quantize_onnx.py --model {WORK}/artifacts/vunalink-mobilenet-v2.onnx --output {WORK}/artifacts/vunalink-mobilenet-v2.int8.onnx --calibration-dir {WORK}/split/validation --test-dir {WORK}/field/test --classes {WORK}/artifacts/model_metadata.json

## 6. Browser reference images and download

In [ ]:
!python ml/create_references.py --model {WORK}/artifacts/model.keras --metadata {WORK}/artifacts/model_metadata.json --test-dir {WORK}/split/test --output-dir {WORK}/artifacts/references
import shutil
for f in ["split_manifest.json"]:
    shutil.copy(f"{WORK}/split/{f}", f"{WORK}/artifacts/{f}")
shutil.copy(f"{WORK}/field/field_manifest.json", f"{WORK}/artifacts/field_manifest.json")
# Ready-made download link, in case the Output panel's menu is hard to reach.
from IPython.display import FileLink, display
os.chdir(WORK); display(FileLink(f"vunalink_model_{MODEL_VERSION}.zip")); os.chdir(f"{WORK}/vunalink")
shutil.make_archive(f"{WORK}/vunalink_model_{MODEL_VERSION}", "zip", f"{WORK}/artifacts")
print(f"Download vunalink_model_{MODEL_VERSION}.zip from the Output panel (/kaggle/working).")

## 7. Put the model in the app

From the zip:

1. Copy `vunalink-mobilenet-v2.onnx`, `class_mapping.json` and `model_metadata.json` to `public/models/`
   (or the `.int8.onnx` renamed to `vunalink-mobilenet-v2.onnx`, if its accuracy held up and it runs in the browser).
2. Copy `evaluation.json`, `evaluation_field.json`, `model_metadata.json`, `class_mapping.json`,
   `vunalink-mobilenet-v2.onnx.metadata.json` and the manifests to `ml/artifacts/`, and `references/` to `ml/evaluation/references/`.
3. In `lib/ml/model.ts`, set `version` to the new `MODEL_VERSION` and `UNSURE_THRESHOLD` to `calibration.unsure_threshold`.
4. Bump `CACHE_NAME` in `public/sw.js` so phones fetch the new model instead of the cached one.
5. Run `npm test`. A test checks that the app's labels, threshold and version match `public/models/model_metadata.json`.

PlantDoc is CC BY 4.0: credit "PlantDoc (Singh et al., 2020)" wherever you show these results.

`model.keras` and `saved_model/` are large and gitignored. Keep them (e.g. as a Kaggle dataset or release asset)
if you want to fine-tune later on your own Rwanda field photos.